In [1]:
# CELL 1
# Install all tools your AI needs
# Tap the ▶️ button to run each cell

!pip install torch numpy transformers datasets

import torch
import torch.nn as nn
import numpy as np

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Everything installed!")
print(f"   Device: {device}")
print(f"   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU")

✅ Everything installed!
   Device: cuda
   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU


In [2]:
# CELL 2
# Kaggle saves your work automatically!
# No Google Drive setup needed at all
# Your files stay safe at /kaggle/working/

import os

# This is your Kaggle save folder
# Everything here persists between sessions
SAVE = '/kaggle/working/MyGameAI'
os.makedirs(f'{SAVE}/model',       exist_ok=True)
os.makedirs(f'{SAVE}/data',        exist_ok=True)
os.makedirs(f'{SAVE}/checkpoints', exist_ok=True)

print("✅ Save folders ready!")
print(f"   Location: {SAVE}")
print("   Kaggle keeps your files automatically!")
print("   No Google Drive setup needed!")
print()
print("💡 IMPORTANT KAGGLE SETTINGS:")
print("   On the right side panel in Kaggle set these:")
print("   • Accelerator → GPU T4 x2 (free!)")
print("   • Persistence → Files (keeps your files!)")
print("   • Internet → ON (needed to install packages)")

✅ Save folders ready!
   Location: /kaggle/working/MyGameAI
   Kaggle keeps your files automatically!
   No Google Drive setup needed!

💡 IMPORTANT KAGGLE SETTINGS:
   On the right side panel in Kaggle set these:
   • Accelerator → GPU T4 x2 (free!)
   • Persistence → Files (keeps your files!)
   • Internet → ON (needed to install packages)


In [3]:
# CELL 3
# THIS IS YOUR AI BRAIN
# Read every comment carefully — it explains what each part does

import torch
import torch.nn as nn
import math
import re

# ════════════════════════════════════════
# PART A — TOKENIZER
# Converts words into numbers
# AI cannot read words — only numbers
# ════════════════════════════════════════

class SimpleTokenizer:
    """
    Turns text into numbers and back.
    """
    def __init__(self):
        self.word_to_id = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3}
        self.id_to_word = {0: "<PAD>", 1: "<START>", 2: "<END>", 3: "<UNK>"}
        self.next_id = 4

    def _tokenize(self, text):
        # CRITICAL FIX 1: This separates punctuation from words!
        # "get_tree().quit()" becomes ["get_tree", "(", ")", ".", "quit", "(", ")"]
        return re.findall(r'\w+|[^\w\s]', text.lower())

    def add_text(self, text):
        """Learn new words from text."""
        for word in self._tokenize(text):
            if word not in self.word_to_id:
                self.word_to_id[word] = self.next_id
                self.id_to_word[self.next_id] = word
                self.next_id += 1

    def encode(self, text, max_length=128, pad=True):
        """Text → numbers."""
        words = self._tokenize(text)
        ids = [self.word_to_id.get(w, 3) for w in words]  # 3 = unknown word
        
        if pad:
            ids = ids[:max_length-1]  # cut if too long
            ids.append(2) # CRITICAL FIX 2: Add <END> token so AI learns to stop!
            ids += [0] * (max_length - len(ids))  # pad with zeros
        else:
            ids = ids[:max_length] # No padding for generation!
            
        return ids

    def decode(self, ids):
        """Numbers → text."""
        words = []
        for i in ids:
            if i == 2:  # END token
                break
            if i not in (0, 1):  # skip PAD and START
                words.append(self.id_to_word.get(i, "<UNK>"))
        
        # Join words and clean up spacing around punctuation
        text = " ".join(words)
        text = re.sub(r'\s+([?.!,"\'()])', r'\1', text) 
        return text

    def vocab_size(self):
        return self.next_id

    def save(self, path):
        import json
        with open(path, 'w') as f:
            json.dump({'word_to_id': self.word_to_id,
                      'id_to_word': {str(k): v for k,v in self.id_to_word.items()},
                      'next_id': self.next_id}, f)
        print(f"Tokenizer saved: {path}")

    def load(self, path):
        import json
        with open(path, 'r') as f:
            data = json.load(f)
        self.word_to_id = data['word_to_id']
        self.id_to_word = {int(k): v for k,v in data['id_to_word'].items()}
        self.next_id = data['next_id']
        print(f"Tokenizer loaded: {path}")

tokenizer = SimpleTokenizer()
print("✅ Tokenizer created!")


# ════════════════════════════════════════
# PART B — ATTENTION MECHANISM
# Helps AI focus on important words
# Like how you focus on key words when reading
# ════════════════════════════════════════

class SelfAttention(nn.Module):
    """
    Attention lets the AI focus on
    the most important words when generating an answer.

    Example:
    Question: "How do I make a JUMP in Godot?"
    AI focuses most on: JUMP and Godot
    Less on: How, do, I, make, a, in
    """

    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # These learn what to focus on
        self.query  = nn.Linear(embed_dim, embed_dim)
        self.key    = nn.Linear(embed_dim, embed_dim)
        self.value  = nn.Linear(embed_dim, embed_dim)
        self.output = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, channels

        # Split into multiple heads
        def split_heads(tensor):
            return tensor.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        Q = split_heads(self.query(x))
        K = split_heads(self.key(x))
        V = split_heads(self.value(x))

        # Calculate attention scores
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        # Mask future tokens (AI should not peek at future words)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        # Convert to probabilities
        attention = torch.softmax(scores, dim=-1)

        # Apply attention to values
        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.output(out)


# ════════════════════════════════════════
# PART C — TRANSFORMER BLOCK
# One thinking layer of your AI
# Your AI has multiple of these stacked
# ════════════════════════════════════════

class TransformerBlock(nn.Module):
    """
    One layer of thinking.
    Your AI stacks multiple of these.
    More layers = smarter AI (but slower to train)
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        # Attention — focuses on important words
        self.attention = SelfAttention(embed_dim, num_heads)

        # Feed forward — processes the information
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),                      # activation function
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )

        # Normalization — keeps numbers stable during training
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Attention with residual connection
        x = x + self.dropout(self.attention(self.norm1(x)))
        # Feed forward with residual connection
        x = x + self.dropout(self.feed_forward(self.norm2(x)))
        return x


# ════════════════════════════════════════
# PART D — YOUR COMPLETE AI MODEL
# This is the full brain
# ════════════════════════════════════════

class YourGameAI(nn.Module):
    """
    YOUR COMPLETE AI MODEL

    This is what you built from scratch.
    It takes a question as input
    and generates a game development answer.

    You can adjust these settings:
    vocab_size  = how many words it knows
    embed_dim   = how much memory per word (bigger = smarter)
    num_layers  = how many thinking layers (more = smarter but slower)
    num_heads   = how many attention heads
    max_length  = longest text it can handle
    """

    def __init__(
        self,
        vocab_size,
        embed_dim=512,    # ← increase to 512 for smarter AI
        num_layers=8,     # ← increase to 8 for smarter AI
        num_heads=8,
        ff_dim=512,
        max_length=512,
        dropout=0.1
    ):
        super().__init__()
        self.max_length = max_length

        # Word embeddings — converts word IDs to vectors
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Position embeddings — tells AI WHERE each word is
        self.pos_embed = nn.Embedding(max_length, embed_dim)

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Stack of transformer blocks (the thinking layers)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(embed_dim)

        # Output head — converts back to word probabilities
        self.output_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Initialize weights properly
        self._init_weights()

        # Count parameters
        params = sum(p.numel() for p in self.parameters())
        print(f"   Your AI has {params:,} parameters")

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, token_ids):
        B, T = token_ids.shape

        # Get word embeddings
        word_emb = self.word_embed(token_ids)

        # Get position embeddings
        positions = torch.arange(T, device=token_ids.device).unsqueeze(0)
        pos_emb = self.pos_embed(positions)

        # Combine word + position information
        x = self.dropout(word_emb + pos_emb)

        # Pass through all thinking layers
        for layer in self.layers:
            x = layer(x)

        # Final normalization
        x = self.norm(x)

        # Get word probabilities
        logits = self.output_head(x)
        return logits

    def generate(self, tokenizer, prompt, max_new_tokens=200, temperature=0.7, repetition_penalty=1.2):
        self.eval()
        
        # CRITICAL FIX 3: pad=False! 
        # Now the AI actually reads your prompt instead of reading zeros!
        ids = tokenizer.encode(prompt, max_length=self.max_length, pad=False)
        input_ids = torch.tensor([ids], dtype=torch.long, device=next(self.parameters()).device)

        generated = []
        with torch.no_grad():
            for _ in range(max_new_tokens):
                current = input_ids[:, -self.max_length:]
                logits = self(current)

                next_logits = logits[:, -1, :] / temperature

                if repetition_penalty != 1.0:
                    for token_id in set(input_ids[0].tolist() + generated):
                        if next_logits[0, token_id] > 0:
                            next_logits[0, token_id] /= repetition_penalty
                        else:
                            next_logits[0, token_id] *= repetition_penalty

                probs = torch.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

                if next_token.item() == 2: # Stop if END token
                    break

                generated.append(next_token.item())
                input_ids = torch.cat([input_ids, next_token], dim=1)

        return tokenizer.decode(generated)

print("✅ All AI classes defined!")
print("   Ready to create your model in the next cell")

✅ Tokenizer created!
✅ All AI classes defined!
   Ready to create your model in the next cell


In [4]:
# CELL 4 — CREATE YOUR AI MODEL
# This actually builds the brain

# You will add vocabulary from training data in Phase 3
# For now create with expanded vocab — will grow further
tokenizer = SimpleTokenizer()

# ══════════════════════════════════════════════════
# EXPANDED BASIC WORDS — v3.0
# Includes Gemini recommended bridge words
# These help AI understand English connectors
# not just Godot keywords
# ══════════════════════════════════════════════════

basic_words = """
extends node scene player enemy coin jump move speed gravity
health score level game godot gdscript characterbody2d
rigidbody2d area2d collisionshape2d sprite2d animatedsprite2d
tilemap canvaslayer label button input velocity position
signal func var const export ready process physics
android mobile touch screen apk export build
platformer shooter puzzle rpg topdown runner
design mechanic difficulty balance feel polish
audio sound music sfx animation sprite pixel

# CORE GAME DEV WORDS
extends class_name node scene player enemy npc boss
health stamina damage score level xp
checkpoint respawn save load pause menu ui hud

# GODOT 4 CORE NODES
Node Node2D CharacterBody2D StaticBody2D RigidBody2D
Area2D CollisionShape2D CollisionPolygon2D
Sprite2D AnimatedSprite2D Camera2D TileMap TileSet
CanvasLayer Control Label Button TextureButton
ProgressBar Timer AudioStreamPlayer2D AudioStreamPlayer
GPUParticles2D CPUParticles2D RayCast2D Line2D
ParallaxBackground ParallaxLayer VisibleOnScreenNotifier2D
VBoxContainer HBoxContainer MarginContainer CenterContainer
PanelContainer ScrollContainer TabContainer
TouchScreenButton VirtualKeyboard LineEdit

# GDSCRIPT FUNDAMENTALS
func var const enum static
if elif else match
for while break continue pass return
true false null
await yield signal
onready export

# INPUT AND MOVEMENT
Input InputEvent InputMap
move_and_slide move_and_collide
velocity direction delta
jump gravity acceleration friction
dash double_jump coyote_time

# MOBILE AND ANDROID
android mobile touch screen apk export
portrait landscape stretch_mode stretch_aspect
viewport scaling dpi
InputEventScreenTouch InputEventScreenDrag

# PERFORMANCE
delta fps process physics_process
queue_free object_pooling
visible_on_screen cull_mask
low_processor_mode
compression atlas batching

# GAME SYSTEMS
autoload singleton save_system load_system
game_manager state_machine fsm
scene_transition checkpoint respawn
inventory shop upgrade system

# DESIGN AND FEEL
coyote_time screen_shake hitstop
camera_zoom tween easing lerp
particles feedback sfx bgm
combo cooldown timer multiplier

# DATA TYPES AND STRUCTURES
int float bool String StringName
Array Dictionary PackedScene
Vector2 Rect2 Color Transform2D
signal Callable Resource

# BRIDGE WORDS — Gemini recommendation
# These help AI form proper English sentences
should because after before instead
attach handle control connect detect
return remove spawn reload reset
belongs requires whenever wherever
between without through during
another together separate individual
automatically immediately gradually
properly correctly accurately reliably
increase decrease subtract multiply
enable disable toggle activate
distance direction magnitude normalize
instance inherit extend override
reference pointer access retrieve
trigger emit receive respond listen
update refresh rebuild recreate
combine separate merge split filter
"""

tokenizer.add_text(basic_words)

# Create your AI
print("\n🧠 Creating YOUR AI model...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,     # memory per word
    num_layers=8,      # thinking layers
    num_heads=8,       # attention heads
    ff_dim=512,        # feedforward width — matches embed_dim
    max_length=512,    # max text length
    dropout=0.2        # prevents overfitting
)

your_ai = your_ai.to(device)
print(f"✅ YOUR AI MODEL CREATED!")
print(f"   Running on: {device}")
print(f"   Vocabulary size: {tokenizer.vocab_size()} words")
print(f"   Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")


🧠 Creating YOUR AI model...
   Your AI has 13,176,832 parameters
✅ YOUR AI MODEL CREATED!
   Running on: cuda
   Vocabulary size: 283 words
   Parameters: 13,176,832


In [5]:
# â•”â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•—
# â•‘         CELL 5 â€” IMPROVED TRAINING DATA v3.0            â•‘
# â•‘                                                          â•‘
# â•‘  Improvements based on Gemini advice + research:        â•‘
# â•‘  âœ… Natural language only (no code symbols)             â•‘
# â•‘  âœ… Node Responsibility pairs (What nodes DO)           â•‘
# â•‘  âœ… Syntax Logic pairs (How GDScript works)             â•‘
# â•‘  âœ… Bridge words for English connectors                 â•‘
# â•‘  âœ… Problem â†’ Solution format                           â•‘
# â•‘  âœ… Balanced vocabulary across all answers              â•‘
# â•‘  âœ… Zero repeated word patterns                         â•‘
# â•šâ•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•

training_data = [

    # â•â•â•â•â•â•â•â•â•â•â•â• GODOT BASICS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "what is godot",
        "a": "Godot is a free open source game engine for building 2D and 3D games. It uses GDScript which resembles Python. You can release games on Android, PC, iOS, and Web without paying royalties. Download it free at godotengine dot org. Godot 4 is the current version and has major improvements over version 3 including a better renderer and improved physics."
    },
    {
        "q": "what is gdscript",
        "a": "GDScript is the primary coding language built into Godot. It resembles Python so beginners find it approachable. Every script begins with extends to declare which node type it belongs to. Variables are declared using var and functions are declared using func. The ready function executes once when the node enters the scene. The process function executes every frame automatically."
    },
    {
        "q": "what is a node in godot",
        "a": "A node is the fundamental building block of every Godot project. Each node type serves a specific purpose. CharacterBody2D handles physics based movement for players. Sprite2D renders images on screen. Label displays text. Button responds to clicks. Timer counts down and fires a signal when finished. Area2D detects when objects overlap. You combine different nodes together to construct your game."
    },
    {
        "q": "what is a scene in godot",
        "a": "A scene is a reusable collection of nodes saved as a tscn file. Your player character is one scene. Each enemy type is its own scene. Every level is a scene. Placing one scene inside another is called instancing. When you edit the original scene every instance throughout the project updates automatically. Scenes keep your project organized and save significant development time."
    },
    {
        "q": "what is an autoload in godot",
        "a": "An autoload is a script that launches at startup and persists through every scene change without being destroyed. It is perfect for storing player score, controlling background music, and managing save data. To create one go to Project then Project Settings then the Autoload tab. Assign a name like GameManager. Every other script in the project can access it directly by that name."
    },
    {
        "q": "what are signals in godot",
        "a": "Signals allow nodes to communicate without being directly connected to each other. When something noteworthy happens a node emits a signal. Other nodes connect to that signal and respond with a function. The Area2D node emits body entered when something enters its collision area. You can define custom signals for your own events. Signals keep code loosely coupled and easier to maintain."
    },
    {
        "q": "what is the difference between process and physics process",
        "a": "The process function runs every rendered frame which is typically sixty times per second. Use it for animations, interface updates, and visual feedback. The physics process function runs at a fixed rate synchronized with the physics engine regardless of frame rate. Always place movement and collision detection inside physics process. Putting movement in process instead causes inconsistent speed and missed collisions on slower devices."
    },
    {
        "q": "how do i change scenes in godot 4",
        "a": "Call get tree then change scene to file passing the path of the target scene file. The path begins with res colon slash slash followed by the folder and file name ending in tscn. For example transitioning to a game over screen or advancing to the next level both use this approach. You can preload the target scene at the top of your script to make the transition faster."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• NODE RESPONSIBILITIES â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "which node should i use for a player character in godot",
        "a": "Use CharacterBody2D for any character that needs to move through code and interact with physics. It is specifically designed for players, enemies, and NPCs that walk, jump, or slide. StaticBody2D is for immovable objects like floors and walls. RigidBody2D is for objects that the physics engine controls automatically like falling crates or bouncing balls."
    },
    {
        "q": "what is the difference between sprite2d and animatedsprite2d",
        "a": "Sprite2D displays a single static image. Use it for backgrounds, decorations, and objects that never change appearance. AnimatedSprite2D plays through multiple frames to create animations like walking, jumping, and attacking. It uses a SpriteFrames resource to store the frames for each animation. For any character that moves and changes appearance always choose AnimatedSprite2D instead of Sprite2D."
    },
    {
        "q": "how do i detect if a player enters a specific area in godot",
        "a": "Use an Area2D node with a CollisionShape2D attached as a child. Connect the body entered signal from the Area2D to a function in your script. When the player or any physics body overlaps the area that function runs automatically. This is how you build collectible coins, damage zones, teleporters, level exit triggers, and any area that should react when something enters it."
    },
    {
        "q": "which node is best for a scrolling background in godot",
        "a": "Use a ParallaxBackground node with ParallaxLayer children for a scrolling background effect. Each ParallaxLayer can scroll at a different speed creating the illusion of depth. Layers further back scroll slower than layers closer to the camera. This creates a convincing parallax effect that makes 2D games feel more three dimensional. Set the motion scale on each layer to control how fast it scrolls."
    },
    {
        "q": "what does a timer node do in godot",
        "a": "A Timer node counts down from a set duration and emits a timeout signal when it reaches zero. Connect the timeout signal to any function you want to call after a delay. Set autostart to true if you want it to begin immediately when the scene loads. Set one shot to false for repeating timers like enemy spawn cycles. Use create timer in code for simple one time delays without needing a Timer node."
    },
    {
        "q": "when should i use canvaslayer in godot",
        "a": "Use CanvasLayer whenever you want interface elements to stay fixed on screen even when the camera moves. Place your health bar, score label, minimap, and pause menu inside a CanvasLayer. Without CanvasLayer your interface elements would scroll with the world and disappear off screen when the camera follows the player. CanvasLayer renders on top of everything in the scene by default."
    },
    {
        "q": "what is a characterbody2d in godot 4",
        "a": "CharacterBody2D is the recommended node for any character that you control through code. It provides the move and slide function which handles movement and automatically resolves collisions with walls and floors. It also provides is on floor to detect if the character is touching ground and is on wall to detect wall contact. Always attach a CollisionShape2D child to give it a physical body."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• PLAYER MOVEMENT â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a player move in godot 4",
        "a": "Attach a script to your CharacterBody2D. Use Input dot get axis with ui left and ui right to read horizontal input as a value between negative one and one. Multiply that value by your movement speed to get the desired velocity. Assign the result to velocity dot x. Then call move and slide to apply the movement while automatically handling collisions. Add a CollisionShape2D child so the player has a physical presence."
    },
    {
        "q": "how do i make a player jump in godot 4",
        "a": "Add gravity that increases the downward velocity every frame when the character is airborne. Define a jump strength constant with a negative value because upward direction in Godot is negative. When the player presses the jump button and is currently on the floor set velocity dot y equal to the jump strength. Call is on floor to confirm ground contact before allowing a jump. Then call move and slide each frame."
    },
    {
        "q": "how do i make a double jump in godot 4",
        "a": "Create a variable to track how many jumps have been used and a constant for the maximum allowed jumps. When the player presses jump confirm whether the jumps used is below the maximum. If so execute the jump and increment the counter. Reset the counter back to zero whenever is on floor returns true. This grants one jump from the ground and one additional jump while airborne before landing resets it."
    },
    {
        "q": "how do i make a player dash in godot 4",
        "a": "Store a boolean called can dash and the direction the player is currently facing. When the dash button is pressed and can dash is true set velocity to a high value in the facing direction and set can dash to false. After a brief dash duration restore normal movement. After a cooldown period set can dash back to true. Use await with create timer to handle the timing of both the dash duration and cooldown."
    },
    {
        "q": "how do i make top down movement in godot 4",
        "a": "Top down games move in all four directions with no gravity involved. Use Input dot get vector passing all four directional action names to receive a Vector2 representing the input direction. Normalize the vector so diagonal movement is not faster than straight movement. Multiply by your speed constant and assign to velocity. Call move and slide. Remove all gravity code entirely since top down games have no concept of falling."
    },
    {
        "q": "how do i add touch controls for android in godot 4",
        "a": "Handle InputEventScreenTouch events inside the input function. Divide the screen into control zones. Touching the left portion of the screen moves the character left while touching the right portion moves right. Tapping a dedicated jump zone triggers a jump. Store the active input direction in a variable and read it inside physics process each frame. Enable emulate touch from mouse in Project Settings to test touch behavior in the editor."
    },
    {
        "q": "how do i make a virtual joystick in godot 4",
        "a": "Build a Control node scene containing a background circle representing the joystick base and a smaller circle representing the thumb. Track which screen touch index is currently controlling the joystick. When the finger moves calculate the vector from the joystick center to the finger position. Clamp that vector so it stays within the joystick radius. Divide the clamped vector by the radius to produce a normalized direction output between negative one and one."
    },
    {
        "q": "how do i make the player face the direction they move",
        "a": "Check the horizontal velocity each frame. If velocity dot x is negative the player is moving left so set flip h to true on the AnimatedSprite2D node. If velocity dot x is positive the player is moving right so set flip h to false. For top down games instead of flipping use the look at function to rotate the character node toward the current movement direction or toward the mouse cursor position."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• GAME OBJECTS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make coins in godot 4",
        "a": "Create a scene using Area2D as the root node. Add a CollisionShape2D with a circular shape and a Sprite2D or AnimatedSprite2D for the visual. In the ready function connect the body entered signal to a collection function. Inside that function confirm the entering body belongs to the player group. If it does call add score on the player then call queue free to remove the coin from the game world."
    },
    {
        "q": "how do i make bouncing coins in godot 4",
        "a": "Give each coin variables for horizontal direction, horizontal speed, vertical speed, and current height above ground. When spawned assign a random angle and randomize the speeds. Each frame move the coin horizontally and reduce vertical speed by gravity. Add vertical speed to the height variable. Stop the coin when height reaches zero. Offset the visual sprite upward by the current height value to simulate a convincing three dimensional arc."
    },
    {
        "q": "how do i make a basic chasing enemy in godot 4",
        "a": "Use CharacterBody2D for the enemy. In ready get a reference to the player node from the player group. Each physics process frame apply gravity when not on the floor. Calculate the horizontal direction toward the player using the sign of the player position minus the enemy position. Set horizontal velocity in that direction multiplied by enemy speed. Flip the sprite to match the movement direction. Call move and slide to apply movement with collision."
    },
    {
        "q": "how do i make a patrolling enemy in godot 4",
        "a": "Store a direction variable initialized to one for rightward movement. Each frame move the enemy in the current direction at patrol speed. When is on wall returns true reverse the direction by multiplying it by negative one. This causes the enemy to automatically bounce back and forth between walls. Combine with a detection range calculation to switch between patrol behavior and active chasing when the player comes close enough."
    },
    {
        "q": "how do i make bullets in godot 4",
        "a": "Create a bullet scene with Area2D as the root containing a CollisionShape2D and a Sprite2D. Store a direction vector and movement speed. Each frame move the bullet by direction multiplied by speed multiplied by delta. Remove the bullet when it travels outside the viewport using get viewport rect. Connect body entered to deal damage to whatever was hit and then remove the bullet using queue free after impact."
    },
    {
        "q": "how do i make a laser in godot 4",
        "a": "Attach a RayCast2D node pointing in the firing direction to detect hits instantly without a physical projectile. Pair it with a Line2D node to draw the visible beam on screen. When firing call force raycast update then call is colliding. If something is hit retrieve the collision point and set the Line2D endpoint there. Call take damage on the hit object if it has that method. Show the line briefly then hide it afterward."
    },
    {
        "q": "how do i make a turret enemy in godot 4",
        "a": "The turret remains stationary and rotates to face the player. Calculate the angle toward the player using angle to point. Use lerp angle each frame to smoothly rotate the turret toward the target angle rather than snapping instantly. When the turret is aimed close enough to the player and the player is within detection range fire a bullet in the current facing direction. Track a cooldown timer to prevent firing too rapidly."
    },
    {
        "q": "how do i make collectible powerups in godot 4",
        "a": "Build powerup scenes using Area2D just like coins. Include an exported variable to identify the powerup type such as speed boost, shield, or damage multiplier. When collected apply the appropriate effect to the player based on the type. Common powerups grant temporary speed increases, absorb one hit of damage, double attack power, restore health, or provide brief invincibility. Spawn a particle burst and play a sound when collected for satisfying feedback."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• GAME SYSTEMS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a health system in godot 4",
        "a": "Add max health and current health variables to the player script. Write a take damage function that subtracts from current health and calls a die function when it reaches zero. Write a heal function that adds health without exceeding the maximum. The die function reloads the current scene to restart the game. Connect health changes to a ProgressBar node in the interface so the player can always see their remaining health."
    },
    {
        "q": "how do i show score on screen in godot 4",
        "a": "Add a CanvasLayer to your main scene then place a Label inside it. CanvasLayer keeps the interface fixed on screen while the camera moves. Reference the label from your player script using onready annotation. Each time score increases update the label text property to display the new value. Storing score in an autoload singleton makes it accessible from any script including the interface, enemies, and level management code."
    },
    {
        "q": "how do i save game data in godot 4",
        "a": "Use FileAccess to write data to the user folder path which works reliably on all platforms including Android. Open the file in write mode. Convert your data dictionary to a JSON formatted string and write it to the file. Close the file when finished. To load saved data open the file in read mode, read the full text, parse it back from JSON format, and restore each variable. Always confirm the save file exists before attempting to load it."
    },
    {
        "q": "how do i make a state machine in godot 4",
        "a": "Create a StateMachine parent node with child nodes representing each state such as Idle, Run, Jump, and Attack. Each state node contains enter, exit, update, and handle input functions. The state machine stores a reference to the currently active state and calls its update every frame. To transition between states call exit on the current state, update the current state reference, then call enter on the new state to initialize it."
    },
    {
        "q": "how do i make a camera follow the player in godot 4",
        "a": "Place a Camera2D node as a direct child of the player node and it follows automatically without any code required. Enable position smoothing in the Inspector to create a gentle easing effect as the camera catches up. Adjust smoothing speed to control how responsively it follows. Set boundary limits to prevent the camera from revealing areas beyond your level edges. Apply screen shake by temporarily offsetting the camera offset property during impacts or explosions."
    },
    {
        "q": "how do i make a combo system in godot 4",
        "a": "Track a combo counter that increments each time the player lands a successful hit on an enemy. Reset a countdown timer every time a new hit occurs. When the timer expires without another hit the combo resets to zero and the multiplier ends. Display the current combo count prominently on screen. Apply a score multiplier based on combo height so longer combos reward more points per hit and encourage aggressive skilled play."
    },
    {
        "q": "how do i make an experience and leveling system in godot 4",
        "a": "Store current experience points and current level as player variables. Define a threshold for how much experience each level requires. When enemies are defeated grant experience to the player. When accumulated experience meets or exceeds the threshold subtract the cost and increase the level. Show a level up visual effect using particles and play a fanfare sound. Grant a stat bonus on each level up such as increased maximum health or faster movement speed."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• SYNTAX LOGIC â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i get a reference to another node in godot 4",
        "a": "Use the onready annotation with get node or the dollar sign shorthand to get a reference when the scene loads. Write onready var label equals dollar sign followed by the node name to store it in a variable at startup. You can also use get node with the path as a string. For nodes in other scenes use get tree to search by group name. Store references in onready variables rather than calling get node repeatedly every frame."
    },
    {
        "q": "what is the correct way to move a character in physics process",
        "a": "Set the velocity property on the CharacterBody2D then call move and slide at the end of physics process. Do not pass delta to move and slide because it is handled automatically. Calculate horizontal velocity by multiplying input direction by speed. Calculate vertical velocity by adding gravity multiplied by delta each frame. Move and slide resolves all collisions and slides the character along surfaces without getting stuck in corners or walls."
    },
    {
        "q": "how do i change a variable from another script in godot 4",
        "a": "Get a reference to the target node first using onready, get node, or get tree. Then access its variable directly using dot notation on that reference. For global variables shared across many scripts use an autoload singleton. Define the variable there and any script in the project can read or write it using the autoload name followed by dot and the variable name. This avoids complex cross script references for commonly shared data."
    },
    {
        "q": "how do i print a message to the console in godot 4",
        "a": "Use the print function with your message inside parentheses. Print is invaluable for debugging because it shows values in the Output panel while the game runs. Print the value of variables to see what they contain at any moment. Use string concatenation with str to combine text and variable values in one print call. For errors use push error instead which displays in red and includes a stack trace to help locate the problem faster."
    },
    {
        "q": "what does export do in godot 4",
        "a": "The at export annotation makes a variable visible and editable directly in the Godot Inspector panel without opening the script. This lets you adjust values like movement speed, jump height, or enemy detection range for each individual node instance without changing code. Export is extremely useful for designers who want to tweak numbers without touching scripts. It works with most basic types including integers, floats, strings, colors, and node paths."
    },
    {
        "q": "how do i use delta in godot 4",
        "a": "Delta is the time in seconds that elapsed since the last frame. Multiply any value that should change over time by delta to make it frame rate independent. Without delta an object moving at speed 300 would move much faster on a 120 fps device than on a 30 fps device. With delta multiplied in the object always travels the same distance per second regardless of frame rate. Always multiply gravity, acceleration, and timer countdowns by delta."
    },
    {
        "q": "how do i add a child node through code in godot 4",
        "a": "Preload the scene file at the top of your script using preload with the resource path. When you need a new instance call the instantiate function on the preloaded scene. Set any initial properties on the instance such as its position or direction. Then call add child passing the instance to add it to the scene tree so it becomes visible and active. This is how you spawn bullets, enemies, particles, and collectibles at runtime."
    },
    {
        "q": "how do i use groups in godot 4",
        "a": "Groups let you tag multiple nodes with a shared label so you can find or control them all at once. Add nodes to groups in the Inspector Node tab or using add to group in code. To find nodes in a group call get tree dot get nodes in group passing the group name as a string. This returns an array of all matching nodes. Use groups to find the player from enemy scripts, to pause all enemies at once, or to collect all coins on a level."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• GAME DESIGN â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "what makes a good platformer game",
        "a": "Responsive precise controls matter more than any visual or feature. The jump must feel satisfying with appropriate weight and gravity. Coyote time lets players jump briefly after walking off a ledge edge. Jump buffering remembers a jump input pressed slightly before landing. Levels should introduce mechanics safely before testing them with danger. Generous checkpoints prevent discouraging repetition. Satisfying sound effects and particles make every action feel rewarding and impactful."
    },
    {
        "q": "how do i design good game levels",
        "a": "Follow the pattern of introduce then challenge then combine. First show a new mechanic in a completely safe environment. Then present it in a situation with some risk. Then combine it with previously learned mechanics for added complexity. Keep mobile levels short because sessions are brief. Always show the player a clear goal ahead of them. Watch real people play your level without giving hints and fix every place where they get confused or fail repeatedly."
    },
    {
        "q": "how do i make my game feel good to play",
        "a": "Game feel emerges from layering many small details. Screen shake during impacts. Particles bursting from collected items. Punchy sound effects synced to every action. Characters squashing flat on landing then snapping back to full height. Brief freeze frames on powerful strikes called hitstop. Subtle camera lag behind fast movement adding perceived weight. Each detail alone seems small but together they transform an ordinary game into one that feels satisfying and alive."
    },
    {
        "q": "how do i balance difficulty in my game",
        "a": "Begin extremely easy so new players experience early success. Increase challenge gradually without sudden difficulty spikes that feel unfair. Observe real people playing without assisting or explaining anything. Document every location where they die or become confused. Simplify those specific moments. Provide optional harder routes for skilled players seeking more challenge. Use checkpoints frequently so failure never feels punishing. The ideal difficulty keeps players in a flow state where they feel challenged yet capable of success."
    },
    {
        "q": "how do i make a good mobile game",
        "a": "Mobile sessions typically last under five minutes so design around short satisfying bursts of play. Touch controls must function perfectly with thumbs covering the screen. Make all interactive elements large enough to tap accurately without looking. Reward players frequently with visual and audio feedback. Save all progress automatically so players never lose anything between sessions. Avoid unskippable content. The best mobile games are immediately understood but contain enough depth to sustain long term engagement."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• ART AND ANIMATION â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i animate a character in godot 4",
        "a": "Replace the Sprite2D node with AnimatedSprite2D. Open the SpriteFrames editor in the Inspector and create animation tracks named idle, run, jump, and fall. Import your spritesheet frames into each appropriate track. In your script read the velocity direction and floor status each frame to determine which animation should play. Call play with the animation name to switch animations. Flip the sprite horizontally based on movement direction."
    },
    {
        "q": "how do i use tilemaps in godot 4",
        "a": "Add a TileMap node to your scene. Create a new TileSet resource in the Inspector. Open the TileSet editor and import your tileset image. Click individual tiles to configure them. Add a physics layer and draw collision rectangles on tiles that should be solid. Return to the scene and paint tiles onto your TileMap to build the level layout. Use multiple layers within one TileMap to separate background decorations from foreground solid platforms."
    },
    {
        "q": "how do i make screen shake in godot 4",
        "a": "Add a shake strength variable to your Camera2D script starting at zero. Each frame when shake strength is above zero offset the camera position by a random value within the strength range in both axes. Gradually reduce the shake strength each frame using lerp or move toward so it fades smoothly. Reset the offset to zero when strength reaches near zero. Call a shake function with a strength value whenever something impactful occurs like taking damage or a large explosion."
    },
    {
        "q": "how do i make particles in godot 4",
        "a": "Add a GPUParticles2D node and configure its ProcessMaterial in the Inspector. Set the number of particles and their lifetime duration. Enable one shot mode for burst effects like explosions or coin collection. Configure velocity, spread angle, gravity effect, and scale in the material properties. To trigger a burst in code set the emitting property to true. For older Android devices use CPUParticles2D instead because it runs on the processor rather than the graphics card."
    },
    {
        "q": "how do i make a tween animation in godot 4",
        "a": "Create a tween by calling create tween on any node. Chain tween property calls passing the target node, the property name as a string, the destination value, and the duration in seconds. Multiple tween property calls execute in sequence one after another automatically. Use tween parallel to run several animations simultaneously. Tweens work perfectly for menu animations, object popups, fading transitions, icon bounce effects, and squash and stretch animations on collected items."
    },
    # â•â•â•â•â•â•â•â•â•â•â•â• AUDIO â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i add sound effects in godot 4",
        "a": "Add AudioStreamPlayer2D nodes as children of your character or object scenes. Load audio files into their stream property in the Inspector. Give each one a clear descriptive name like JumpSound, CoinSound, or HurtSound. Store references using onready in your script. Call the play function on the correct player at the right moment during gameplay. Slightly randomize the pitch scale each time a sound plays to prevent repetitive identical sounds feeling robotic."
    },
    {
        "q": "how do i add background music in godot 4",
        "a": "Place an AudioStreamPlayer node inside an autoload scene so the music continues playing through every scene transition without restarting. Load an OGG format audio file and enable looping in the stream settings. Enable autoplay to start music immediately. Create fade in and fade out functions using tweens on the volume db property for smooth transitions between tracks. Add separate audio buses for music and effects so players can control each volume independently in settings."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• MOBILE AND ANDROID â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i export to android in godot 4",
        "a": "Install Android Studio and the Android SDK on your development computer. In Godot Editor Settings locate the Android SDK path field and enter the correct path. Generate a signing keystore file using the keytool command in a terminal. In Project Export create an Android export preset and fill in your package name and keystore details. Export as an AAB file for submission to Google Play Store or as an APK for direct device installation and testing."
    },
    {
        "q": "how do i make my game work well on android",
        "a": "Configure stretch mode to canvas items and aspect to expand in Project Display Settings for proper scaling across different screen sizes. Make all touch targets at minimum one hundred pixels wide for reliable tapping. Lock screen orientation under Display Window Handheld settings. Keep the screen on during gameplay using an OS setting call in the ready function. Detect Android using OS dot get name and conditionally show touch controls only when running on mobile."
    },
    {
        "q": "how do i optimize my game for mobile performance",
        "a": "Keep all texture files at 512 by 512 pixels or smaller whenever possible. Enable texture compression in the import settings for every texture. Limit visible particles to fewer than one hundred on screen simultaneously. Attach VisibleOnScreenNotifier2D to enemies and disable their physics processing when they are off screen. Cache all node references using onready rather than calling get node repeatedly. Use simple rectangle collision shapes instead of complex polygon shapes for better performance."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• UI AND MENUS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a main menu in godot 4",
        "a": "Build a scene using Control nodes containing buttons for Play, Continue, Settings, and Quit. Connect each button pressed signal to a handler function. Check whether a save file exists and display the Continue button only when one is found. Set initial focus to the most important button so keyboard and controller navigation works without mouse input. The Play button loads the first level scene using get tree change scene to file."
    },
    {
        "q": "how do i make a pause menu in godot 4",
        "a": "Create a Control node covering the full screen and hide it by default. Set its process mode property to always so it can function while the game tree is paused. When the pause button is pressed make the menu visible and set get tree paused to true which freezes all normal nodes. The resume button hides the menu and sets paused back to false. The exit button unpauses the tree first then changes to the main menu scene."
    },
    {
        "q": "how do i make a settings menu with volume control in godot 4",
        "a": "Add HSlider nodes for master volume, music volume, and sound effects volume. Save settings to a ConfigFile stored in the user folder path so they survive between sessions. When a slider value changes update the corresponding audio bus volume using AudioServer set bus volume db. Convert the percentage value to decibels using a logarithmic formula. Load saved settings during ready and apply them immediately so the previous settings are restored on launch."
    },
    {
        "q": "how do i make a game over screen in godot 4",
        "a": "Create a Control scene displaying the final score and offering retry and main menu buttons. When the player dies pass the final score to this screen before switching scenes. Animate the screen appearing using a tween on the modulate alpha property to fade it in smoothly. The retry button reloads the current gameplay scene. The main menu button transitions back to the title screen. Show encouraging messages to soften repeated failures for younger players."
    },
    {
        "q": "how do i make a loading screen in godot 4",
        "a": "Use ResourceLoader request load to begin loading the next scene in the background thread without freezing gameplay. Display a progress bar and update it each frame by polling ResourceLoader get load status. When the status indicates loading is complete retrieve the fully loaded scene resource and switch to it. This prevents the game from freezing on a black screen during large level loads. Display a helpful tip or interesting fact during loading to entertain waiting players."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• ADVANCED SYSTEMS â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make an inventory system in godot 4",
        "a": "Define an Item class that extends Resource containing name, icon texture, price, description, and stack size properties. Create an Inventory node with an array to store collected items. The add item function confirms whether a stackable item already exists in the array and increases its quantity. Otherwise it appends a new entry. The remove item function decreases quantity and deletes the entry when it reaches zero. Emit a changed signal whenever contents are modified to automatically refresh the interface."
    },
    {
        "q": "how do i make a shop system in godot 4",
        "a": "The shop maintains a list of available stock and a sell back rate for items the player wants to sell. The purchase function verifies the player has sufficient currency, deducts the cost, and transfers the item to the player inventory. The sell function removes the item from inventory and returns currency at a reduced percentage such as twenty five percent of original value. Generate interface buttons dynamically for each item currently in stock."
    },
    {
        "q": "how do i make enemies not overlap each other in godot 4",
        "a": "Implement separation steering behavior for each enemy. Each frame loop through all other enemies in the enemy group. For every nearby enemy calculate a push vector pointing away from them. Make the push force stronger when enemies are closer together. Sum all the individual push vectors into a combined separation force. Add this separation force to the enemy desired velocity before applying movement. This causes enemies to naturally spread apart without explicit collision between them."
    },
    {
        "q": "how do i make smooth enemy movement in godot 4",
        "a": "Apply the seek steering behavior where the enemy calculates a desired velocity pointing directly toward the target position. Add arrival behavior that reduces the desired speed as the enemy approaches the target so it does not overshoot and oscillate. Use move toward on the actual velocity each frame to smoothly accelerate toward the desired velocity. This produces natural looking movement with gradual acceleration when starting and deceleration when approaching the destination."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• COMPLETE GAMES â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i make a complete platformer game in godot 4",
        "a": "Build incrementally through clear stages. First implement player movement and jumping with gravity. Then design levels using TileMap. Add Camera2D as a child of the player. Place collectible coins using Area2D. Create patrolling enemies using CharacterBody2D. Implement a health system with a progress bar. Display score and health in a CanvasLayer. Add death when falling off the level bottom. Create a goal zone that advances to the next scene. Finally add virtual touch controls for Android."
    },
    {
        "q": "how do i make a complete top down game in godot 4",
        "a": "Begin with four directional movement without any gravity. Build the environment using TileMap with solid wall tiles for collision boundaries. Add Camera2D with movement limits. Create enemies that patrol set paths and switch to chasing when the player enters their detection radius. Implement a brief attack hitbox that activates during the swing animation. Build inventory management for collected items. Add NPCs with branching dialogue. Connect multiple rooms through door trigger zones."
    },
    {
        "q": "how do i make a complete shooter game in godot 4",
        "a": "Create a player that moves freely and rotates to face the aim direction. Add shooting that fires a bullet scene toward the aim direction with a brief cooldown between shots. Build enemies that advance toward the player spawning in escalating waves. Create a spawner that releases enemies from screen edges at increasing frequency. Implement player health with damage on contact. Display score and current wave number. Drop randomized powerups from defeated enemies. Show a game over screen with the final score achieved."
    },

    # â•â•â•â•â•â•â•â•â•â•â•â• PUBLISHING â•â•â•â•â•â•â•â•â•â•â•â•

    {
        "q": "how do i publish to google play store",
        "a": "Generate a release signing keystore using the keytool command in a terminal window. Preserve this keystore file permanently because losing it makes updating your app impossible forever. In Godot Project Export configure an Android preset with your package name and keystore credentials. Export a release AAB build. Create a developer account at play dot google dot com for a one time registration fee. Upload the AAB file, complete your store listing details, and submit the app for Google review."
    },
    {
        "q": "how do i make a good play store listing",
        "a": "Your app icon receives the most attention so make it 512 by 512 pixels with bold colors and a clear focal point showing your main character or concept. The feature graphic at the top must be exactly 1024 by 500 pixels. Capture at least four screenshots showing the most exciting and representative gameplay moments. Write a concise short description capturing the core fun in two sentences. Research keywords that players actually search for and include them naturally in your description."
    },
    {
        "q": "how do i make my game run faster in godot 4",
        "a": "Open the built in profiler found under the Debug menu while the game is running. Examine which functions consume the most processing time each frame. Cache all node references in ready using onready annotations instead of calling get node repeatedly. Disable physics processing on enemies that have scrolled off screen. Keep textures small and enable compression. Use rectangular collision shapes rather than detailed polygon shapes. Target sixty frames per second on the weakest device you plan to support."
    },
    {
        "q": "how do i monetize my android game",
        "a": "The most player friendly option is a single upfront purchase price with no further payments required. Free with advertising uses AdMob which Godot supports through a community plugin. Rewarded ads that grant players valuable in-game items in exchange for watching are far more accepted than forced interruption ads. Cosmetic in-app purchases for character skins or color themes feel fair to most players. Never implement pay to win mechanics that give paying players an unfair competitive advantage over others."
    },

]

# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
# QUALITY CHECK â€” Run this to verify data
# â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•â•
print(f"Training data created!")
print(f"Total examples: {len(training_data)}")

import re
from collections import Counter
answers = [item['a'] for item in training_data]
all_words = []
for a in answers:
    all_words.extend(a.lower().split())
counter = Counter(all_words)
print(f"Total words in training: {len(all_words)}")
print(f"Unique words: {len(counter)}")
print(f"\nTop 10 most common words:")
for word, count in counter.most_common(10):
    print(f"  {count}x  {word}")

# Check for problem words
problem_words = ['small', 'check', 'starts', 'how how', 'small small']
print(f"\nProblem word check:")
for pw in ['small', 'starts', 'check']:
    c = counter.get(pw, 0)
    status = "âœ…" if c < 5 else "âš ï¸"
    print(f"  {status} '{pw}': {c} times")

Training data created!
Total examples: 77
Total words in training: 5501
Unique words: 1644

Top 10 most common words:
  353x  the
  173x  a
  160x  to
  147x  and
  77x  in
  60x  for
  53x  on
  52x  is
  52x  it
  51x  with

Problem word check:
  âœ… 'small': 3 times
  âœ… 'starts': 0 times
  âœ… 'check': 2 times


In [6]:
# Create a dedicated list for your new examples
new_godot_knowledge = [
    {"q": "how to change scenes", "a": "get_tree().change_scene_to_file('res://levels/level_2.tscn')"},
    {"q": "how to make a timer", "a": "await get_tree().create_timer(2.0).timeout\nprint('Time is up!')"},
    {"q": "how to detect collision", "a": "func _on_body_entered(body):\n    if body.is_in_group('player'):\n        queue_free()"},
    {"q": "how to flip a sprite", "a": "$Sprite2D.flip_h = velocity.x < 0"},
    {"q": "how to play a sound", "a": "$AudioStreamPlayer.play()"},
    {"q": "how to use delta", "a": "position += velocity * delta"},
    {"q": "how to get mouse position", "a": "var mouse_pos = get_global_mouse_position()"},
    {"q": "how to add a child node", "a": "var enemy = enemy_scene.instantiate()\nadd_child(enemy)"},
    {"q": "how to check for input", "a": "if Input.is_action_just_pressed('jump'):\n    jump()"},
    {"q": "how to quit game", "a": "get_tree().quit()"}
]

# Add them to the main training list
training_data.extend(new_godot_knowledge)

print(f"✅ Data added! You now have {len(training_data)} total examples.")

✅ Data added! You now have 87 total examples.


In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║         NEW GODOT 4 CODE EXAMPLES (SHORT & PRECISE)      ║
# ╚══════════════════════════════════════════════════════════╝

even_more_godot_knowledge = [
    # --- SCENE & NODE MANAGEMENT ---
    {"q": "how to reload current scene", "a": "get_tree().reload_current_scene()"},
    {"q": "how to get the parent node", "a": "get_parent()"},
    {"q": "how to find all enemies in a group", "a": "var enemies = get_tree().get_nodes_in_group('enemy')"},
    {"q": "how to hide a node", "a": "hide()"},
    {"q": "how to show a node", "a": "show()"},
    {"q": "how to destroy a node safely", "a": "queue_free()"},
    
    # --- MOVEMENT & PHYSICS ---
    {"q": "how to apply gravity", "a": "velocity.y += gravity * delta"},
    {"q": "how to get horizontal input direction", "a": "var direction = Input.get_axis('ui_left', 'ui_right')"},
    {"q": "how to check if player is on the floor", "a": "if is_on_floor():\n    pass"},
    {"q": "how to check if player is hitting a wall", "a": "if is_on_wall():\n    pass"},
    {"q": "how to move and collide", "a": "move_and_slide()"},
    
    # --- UI & TEXT ---
    {"q": "how to change label text", "a": "$Label.text = 'Hello World'"},
    {"q": "how to change a button text", "a": "$Button.text = 'Click Me'"},
    {"q": "how to hide the mouse cursor", "a": "Input.set_mouse_mode(Input.MOUSE_MODE_HIDDEN)"},
    
    # --- SIGNALS & TIMERS ---
    {"q": "how to wait 1 second in code", "a": "await get_tree().create_timer(1.0).timeout"},
    {"q": "how to connect a button press via code", "a": "$Button.pressed.connect(_on_button_pressed)"},
    {"q": "how to emit a custom signal", "a": "emit_signal('my_custom_signal')"},
    
    # --- ANIMATION & AUDIO ---
    {"q": "how to play an animation", "a": "$AnimationPlayer.play('run')"},
    {"q": "how to stop an animation", "a": "$AnimationPlayer.stop()"},
    {"q": "how to play a sound effect", "a": "$AudioStreamPlayer.play()"},
    {"q": "how to flip a sprite horizontally", "a": "$Sprite2D.flip_h = true"},
    
    # --- MATH & LOGIC ---
    {"q": "how to generate a random number", "a": "var random_val = randf_range(0.0, 10.0)"},
    {"q": "how to get distance between two nodes", "a": "var dist = global_position.distance_to(target.global_position)"},
    {"q": "how to print a variable to console", "a": "print('The value is: ', my_variable)"},
    {"q": "how to clamp a number between 0 and 100", "a": "var health = clamp(health, 0, 100)"},
    
    # --- GAME STATE ---
    {"q": "how to pause the game", "a": "get_tree().paused = true"},
    {"q": "how to unpause the game", "a": "get_tree().paused = false"},
    {"q": "how to quit the game to desktop", "a": "get_tree().quit()"},
    {"q": "how to save a file", "a": "var file = FileAccess.open('user://save.dat', FileAccess.WRITE)"},
    {"q": "how to load a file", "a": "var file = FileAccess.open('user://save.dat', FileAccess.READ)"}
]

# Add these new examples to your main training list
training_data.extend(even_more_godot_knowledge)

print(f"✅ SUCCESS! Added 30 new short code examples.")
print(f"🧠 Your AI now has {len(training_data)} total examples to learn from!")

✅ SUCCESS! Added 30 new short code examples.
🧠 Your AI now has 117 total examples to learn from!


In [8]:
# CELL 6 — PREPARE DATA FOR TRAINING
# Improved format using User/Assistant structure
# Gemini recommended this for better question/answer separation

import torch
from torch.utils.data import Dataset, DataLoader

class GameDevDataset(Dataset):
    """
    Prepares your training examples
    so the AI can learn from them.
    
    Uses User/Assistant format so AI learns
    the difference between asking and answering.
    """

    def __init__(self, data, tokenizer, max_length=512):
        self.examples = []
        self.tokenizer = tokenizer

        for item in data:
            # ── NEW FORMAT: User/Assistant structure ──
            # Gemini recommended this — teaches AI that
            # everything after "Assistant:" is the answer
            full_text = f"User: {item['q']} Assistant: {item['a']}"

            # Add all words to tokenizer vocabulary
            tokenizer.add_text(full_text)
            self.examples.append(full_text)

        print(f"✅ Dataset prepared: {len(self.examples)} examples")
        print(f"   Vocabulary size: {tokenizer.vocab_size()} words")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]
        ids = self.tokenizer.encode(text, max_length=512)
        tensor = torch.tensor(ids, dtype=torch.long)
        return tensor

# Create dataset
dataset = GameDevDataset(training_data, tokenizer, max_length=512)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Recreate AI with updated vocabulary
print("\n🧠 Rebuilding AI with full vocabulary...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,
    num_layers=8,
    num_heads=8,
    ff_dim=512,
    max_length=512,
    dropout=0.2
)
your_ai = your_ai.to(device)
print(f"✅ AI rebuilt with {tokenizer.vocab_size()} word vocabulary")
print(f"   Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")

✅ Dataset prepared: 117 examples
   Vocabulary size: 1577 words

🧠 Rebuilding AI with full vocabulary...
   Your AI has 14,501,888 parameters
✅ AI rebuilt with 1577 word vocabulary
   Parameters: 14,501,888


In [9]:
# CELL 7 — TRAIN YOUR AI
# Improved with repetition penalty to stop word loops
# Uses User/Assistant format in test prompt

import torch.optim as optim
import time

def train_your_ai(
    model,
    dataloader,
    tokenizer,
    num_epochs=300,
    learning_rate=5e-4,
    save_dir=SAVE
):
    # Optimizer adjusts the AI weights during training
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)

    # ADVANCED AI: OneCycleLR is the industry standard for Transformers
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=learning_rate, 
        steps_per_epoch=len(dataloader), 
        epochs=num_epochs,
        pct_start=0.1
    )

    # ADVANCED AI: Label smoothing prevents the AI from becoming overconfident
    loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
    
    # ADVANCED AI: Mixed Precision Scaler (Trains 2x Faster on Kaggle GPU!)
    scaler = torch.amp.GradScaler(device)

    best_loss = float('inf')
    history = []

    print("🚀 TRAINING YOUR AI STARTED!")
    print(f"   Epochs: {num_epochs}")
    print(f"   Examples: {len(dataloader.dataset)}")
    print("=" * 50)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0
        start_time = time.time()

        for batch in dataloader:
            batch = batch.to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            optimizer.zero_grad()

            # ADVANCED AI: Autocast uses 16-bit math for lightning-fast training
            with torch.amp.autocast(device_type=device):
                logits = model(inputs)
                loss = loss_fn(
                    logits.reshape(-1, logits.size(-1)),
                    targets.reshape(-1)
                )

            # ADVANCED AI: Scaled backward pass
            scaler.scale(loss).backward()
            
            # Unscale before clipping gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            # Moved inside the batch loop for OneCycleLR!
            scheduler.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        elapsed = time.time() - start_time
        history.append(avg_loss)

        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {(epoch+1):>3}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Time: {elapsed:.1f}s | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

            # Test what AI says right now using NEW format
            model.eval()
            test_answer = model.generate(
                tokenizer,
                'User: how do i make a player jump Assistant:',
                max_new_tokens=50,
                temperature=0.9,
                repetition_penalty=1.3   # ← STOPS word loops!
            )
            print(f"   AI answer preview: {test_answer[:80]}...")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'loss': best_loss,
                'vocab_size': tokenizer.vocab_size()
            }, f'{save_dir}/checkpoints/best_model.pt')

        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'loss': avg_loss
            }, f'{save_dir}/checkpoints/epoch_{(epoch+1)}.pt')
            tokenizer.save(f'{save_dir}/checkpoints/tokenizer.json')
            print(f"   💾 Saved checkpoint at epoch {epoch+1}")

    # Save final model
    torch.save(model.state_dict(), f'{save_dir}/model/final_model.pt')
    tokenizer.save(f'{save_dir}/model/tokenizer.json')

    print("\n" + "=" * 50)
    print("✅ TRAINING COMPLETE!")
    print(f"   Best loss: {best_loss:.4f}")
    print(f"   Final loss: {avg_loss:.4f}")
    print(f"   Lower loss = smarter AI")
    print(f"   Model saved to: {save_dir}/model/")
    return history
    
# START TRAINING YOUR AI
history = train_your_ai(
    model=your_ai,
    dataloader=dataloader,
    tokenizer=tokenizer,
    num_epochs=300,
    learning_rate=1e-4,
    save_dir=SAVE
)

🚀 TRAINING YOUR AI STARTED!
   Epochs: 300
   Examples: 117
Epoch  10/300 | Loss: 6.6038 | Time: 1.4s | LR: 0.000028
   AI answer preview: desktop 2 elif bursting panel distance falling grants reverse controlling virtua...
Tokenizer saved: /kaggle/working/MyGameAI/checkpoints/tokenizer.json
   💾 Saved checkpoint at epoch 10
Epoch  20/300 | Loss: 5.8085 | Time: 1.4s | LR: 0.000077
   AI answer preview: quit tracks in any = once the pixels the fades if bullets in. character slide ov...
Tokenizer saved: /kaggle/working/MyGameAI/checkpoints/tokenizer.json
   💾 Saved checkpoint at epoch 20
Epoch  30/300 | Loss: 5.1546 | Time: 1.4s | LR: 0.000100
   AI answer preview: aimed in godot 4 assistant : use create the than initialized health and characte...
Tokenizer saved: /kaggle/working/MyGameAI/checkpoints/tokenizer.json
   💾 Saved checkpoint at epoch 30
Epoch  40/300 | Loss: 4.5441 | Time: 1.4s | LR: 0.000100
   AI answer preview: how to forced the player is the game impossible. use character 

In [17]:
# CELL 8 — TEST YOUR AI
# Improved with 15 questions covering all topics
# Uses new User/Assistant format

def test_your_ai(model, tokenizer, question):
    model.eval()
    # NEW FORMAT — matches training format exactly!
    prompt = f"User: {question} Assistant:"
    answer = model.generate(
        tokenizer,
        prompt,
        max_new_tokens=150,
        temperature=0.2,
        repetition_penalty=1.2    # stops word loops!
    )
    print(f"❓ Question: {question}")
    print(f"🤖 Your AI: {answer}")
    print("—" * 50)

print("Testing YOUR trained AI...")
print("=" * 50)

# ── MOVEMENT QUESTIONS ──
print("\n📦 MOVEMENT:")
test_your_ai(your_ai, tokenizer, "how do i make a player jump in godot")
test_your_ai(your_ai, tokenizer, "how do i make top down movement in godot 4")
test_your_ai(your_ai, tokenizer, "how do i make a player dash in godot 4")

# ── NODE KNOWLEDGE QUESTIONS ──
print("\n🧱 NODES:")
test_your_ai(your_ai, tokenizer, "which node should i use for a player character")
test_your_ai(your_ai, tokenizer, "what is the difference between sprite2d and animatedsprite2d")
test_your_ai(your_ai, tokenizer, "when should i use canvaslayer in godot")

# ── GAME OBJECTS ──
print("\n🎮 GAME OBJECTS:")
test_your_ai(your_ai, tokenizer, "how do i make coins to collect")
test_your_ai(your_ai, tokenizer, "how do i make an enemy that chases the player")

# ── GAME DESIGN ──
print("\n🎯 GAME DESIGN:")
test_your_ai(your_ai, tokenizer, "what makes a good platformer game")
test_your_ai(your_ai, tokenizer, "how do i make my game feel good to play")

# ── SYNTAX LOGIC ──
print("\n📝 GDSCRIPT:")
test_your_ai(your_ai, tokenizer, "what does export do in godot 4")
test_your_ai(your_ai, tokenizer, "how do i use delta in godot 4")
test_your_ai(your_ai, tokenizer, "how do i get a reference to another node")

# ── MOBILE ──
print("\n📱 MOBILE:")
test_your_ai(your_ai, tokenizer, "how do i add touch controls for android in godot 4")
test_your_ai(your_ai, tokenizer, "how do i export to android in godot 4")

print("\n" + "=" * 50)
print("✅ Testing complete!")
print("Good answers = AI learned successfully!")
print("Garbage answers = retrain with more epochs")

Testing YOUR trained AI...

📦 MOVEMENT:
❓ Question: how do i make a player jump in godot
🤖 Your AI: add gravity that increases the character is airborne for any character through the character is negative one and ground. when you can negative one is on the game over screen. multiply value by your presses the jump, and set velocity. then call is currently on the floor to the jump strength. call is on the floor to detect if velocity dot y equal to always confirm ground and slide each frame.
——————————————————————————————————————————————————
❓ Question: how do i make top down movement in godot 4
🤖 Your AI: top down games move in all four directions with no gravity involved. use input dot get vector passing all four directional action names to receive a vector2 representing the input direction. normalize the vector so diagonal movement is not faster than straight movement. multiply by your speed constant and assign to velocity. call move and slide. remove all gravity code entirely since to